In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
sns.set_theme(
    style="whitegrid",
    context="notebook")

sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = (10, 6)

plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 15
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['figure.autolayout'] = True

In [ ]:
df = pd.read_excel(r"C:\Users\amare\OneDrive\Desktop\Crime Hotspots Capstone Project\crime-hotspot-prediction-project\data\03_primary\master_dataset.xlsx")


In [ ]:
print(f"Number of Missing Values in the Master Dataset: \n{df.isna().sum()}")

In [ ]:
print(f"Number of Duplicates in the Master Dataset: {df.duplicated().sum()}")

In [ ]:
df["year"] = df["date"].dt.year


In [ ]:
sns.lineplot(data = df, x = "year", y = "Crime Count", errorbar=('ci', False), hue = "Cluster")

In [ ]:
sns.histplot(df, x = "Crime Count")

In [ ]:
# scale for KNN then leave as if for tree based models
import numpy as np
import sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

df["log_Crime Count"] = np.log1p(df["Crime Count"])
df["crime_scaled"] = scaler.fit_transform(df[["log_Crime Count"]])

In [ ]:
# total number of elements in Cluster column/ Class Balance
df.groupby("Cluster").size().sort_values(ascending=False)

In [ ]:
# --- Lag features (previous year values per Cluster) ---
lag_cols = ["Crime Count", "population_density", "population_unemployment", "poor_households"]
for col in lag_cols:
    df[f"{col}_lag1"] = df.groupby("Cluster")[col].shift(1)
    df[f"{col}_lag2"] = df.groupby("Cluster")[col].shift(2)

# --- Rolling statistics (3-year window per Cluster) ---
for col in lag_cols:
    df[f"{col}_rolling_mean_3"] = (
        df.groupby("Cluster")[col]
        .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
    )
    df[f"{col}_rolling_std_3"] = (
        df.groupby("Cluster")[col]
        .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).std())
    )

# --- Year-over-year change ---
for col in lag_cols:
    df[f"{col}_yoy_change"] = df.groupby("Cluster")[col].pct_change()
